In [ ]:
import os
import sys
from pathlib import Path

BASE = Path.home() / "BigData_UPAO" / "proyecto_afp_v2"

os.environ["HADOOP_HOME"] = "/opt/hadoop-3.3.6"
os.environ["HADOOP_CONF_DIR"] = "/opt/hadoop-3.3.6/etc/hadoop"
os.environ["YARN_CONF_DIR"] = os.environ["HADOOP_CONF_DIR"]

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "10.0.2.15"

BASE.joinpath("evidencias").mkdir(exist_ok=True)

print("Proyecto:", BASE)
print("Carpeta de evidencias:", BASE / "evidencias")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

derby_path = (
    BASE / "metastore" / "metastore_db"
).as_posix()

spark = (
    SparkSession.builder
    .appName("afp_v2_verificaciones_finales")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .config("spark.driver.host", "10.0.2.15")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.instances", "1")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "512m")
    .config(
        "javax.jdo.option.ConnectionURL",
        f"jdbc:derby:;databaseName={derby_path};create=true"
    )
    .config(
        "spark.hadoop.javax.jdo.option.ConnectionURL",
        f"jdbc:derby:;databaseName={derby_path};create=true"
    )
    .config(
        "spark.sql.warehouse.dir",
        "hdfs://localhost:9000/user/junior/datalake_afp_v2/catalog/warehouse"
    )
    .enableHiveSupport()
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark:", spark.version)
print("MASTER:", spark.sparkContext.master)
print("CATALOGO:", spark.conf.get("spark.sql.catalogImplementation"))

In [ ]:
final = spark.table(
    "afp_analytics.tabla_analitica_mensual"
)

memorias = spark.table(
    "afp_analytics.memorias_indicadores"
)

afiliados = spark.table(
    "afp_analytics.sbs_afiliados"
)

rentabilidad = spark.table(
    "afp_analytics.bcrp_json_rentabilidad"
)

print("Tabla analítica:", final.count())
print("Memorias:", memorias.count())
print("Afiliados:", afiliados.count())
print("Rentabilidad BCRP:", rentabilidad.count())

In [ ]:
columnas_nulos = [
    "anio_ejercicio",
    "afiliados_integra"
]

for columna in columnas_nulos:
    print(f"\n=== NULOS EN {columna} ===")

    final.filter(
        F.col(columna).isNull()
    ).select(
        "fecha_mes",
        columna
    ).orderBy("fecha_mes").show(30, truncate=False)

In [ ]:
print("COBERTURA ORIGINAL DE AFILIADOS")

afiliados.filter(
    F.lower(F.col("afp")) == "integra"
).agg(
    F.min("fecha_mes").alias("fecha_minima"),
    F.max("fecha_mes").alias("fecha_maxima"),
    F.count("*").alias("registros"),
    F.sum(
        F.when(F.col("afiliados").isNull(), 1).otherwise(0)
    ).alias("valores_nulos")
).show()

print("MESES SIN AFILIADOS EN TABLA FINAL")

final.filter(
    F.col("afiliados_integra").isNull()
).select("fecha_mes").orderBy("fecha_mes").show(20)

In [ ]:
from pyspark.sql import functions as F

print("=== DISPONIBILIDAD DE MEMORIAS ===")

memorias.select(
    "afp",
    "anio_ejercicio",
    "parte",
    "disponible_desde",
    "caracteres"
).orderBy(
    "afp",
    "anio_ejercicio",
    "parte"
).show(50, truncate=False)

print("=== FECHAS DE DISPONIBILIDAD NULAS ===")

nulos_disponibilidad = memorias.filter(
    F.col("disponible_desde").isNull()
).count()

print("Total de fechas nulas:", nulos_disponibilidad)

In [ ]:
print("=== VERIFICACIÓN DE INFORMACIÓN FUTURA ===")

futuras = final.filter(
    F.col("disponible_desde").isNotNull()
    &
    (
        F.col("disponible_desde")
        > F.last_day(F.col("fecha_mes"))
    )
)

print(
    "Total de registros con información futura:",
    futuras.count()
)

futuras.select(
    "fecha_mes",
    "anio_ejercicio",
    "disponible_desde"
).orderBy(
    "fecha_mes"
).show(30, truncate=False)

In [ ]:
print("=== ESTADÍSTICAS DESCRIPTIVAS ===")

variables = [
    "rent_real_integra_12m",
    "rent_real_habitat_12m",
    "rent_real_prima_12m",
    "rent_real_profuturo_12m",
    "ipc_lag1",
    "tipo_cambio_lag1"
]

final.select(*variables).summary(
    "count",
    "min",
    "25%",
    "50%",
    "75%",
    "max",
    "mean",
    "stddev"
).show(truncate=False)

In [ ]:
print("=== DETECCIÓN DE VALORES ATÍPICOS (IQR) ===")

resultados_atipicos = []

for nombre in variables:

    datos = final.select(
        F.col(nombre).cast("double").alias("valor")
    ).filter(
        F.col("valor").isNotNull()
    )

    q1, q3 = datos.approxQuantile(
        "valor",
        [0.25, 0.75],
        0.001
    )

    iqr = q3 - q1

    inferior = q1 - 1.5 * iqr
    superior = q3 + 1.5 * iqr

    total = datos.filter(
        (F.col("valor") < inferior)
        |
        (F.col("valor") > superior)
    ).count()

    resultados_atipicos.append({
        "variable": nombre,
        "limite_inferior": inferior,
        "limite_superior": superior,
        "posibles_atipicos": total
    })

    print(
        nombre,
        "| Límite inferior:", round(inferior, 4),
        "| Límite superior:", round(superior, 4),
        "| Posibles atípicos:", total
    )

In [ ]:
print("=== RENTABILIDADES MÁS BAJAS ===")

nombre = "rent_real_integra_12m"

final.select(
    "fecha_mes",
    nombre
).filter(
    F.col(nombre).isNotNull()
).orderBy(
    F.col(nombre).asc()
).show(10, truncate=False)


print("=== RENTABILIDADES MÁS ALTAS ===")

final.select(
    "fecha_mes",
    nombre
).filter(
    F.col(nombre).isNotNull()
).orderBy(
    F.col(nombre).desc()
).show(10, truncate=False)

In [ ]:
print("=== DISTRIBUCIÓN DE LA VARIABLE OBJETIVO ===")

final.groupBy(
    "objetivo_bajo_rendimiento_mes_siguiente"
).count().orderBy(
    "objetivo_bajo_rendimiento_mes_siguiente"
).show()


print("=== MESES SIN ETIQUETA ===")

final.filter(
    F.col(
        "objetivo_bajo_rendimiento_mes_siguiente"
    ).isNull()
).select(
    "fecha_mes"
).show()

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w = Window.orderBy("fecha_mes")

auditoria = (
    final.select(
        "fecha_mes",
        "brecha_real_pp",
        "objetivo_bajo_rendimiento_mes_siguiente"
    )
    .withColumn(
        "brecha_siguiente",
        F.lead("brecha_real_pp").over(w)
    )
    .withColumn(
        "objetivo_esperado",
        F.when(
            F.col("brecha_siguiente").isNull(),
            F.lit(None).cast("int")
        ).otherwise(
            F.when(
                F.col("brecha_siguiente") < 0, 1
            ).otherwise(0)
        )
    )
)

diferencias = auditoria.filter(
    ~F.col("objetivo_bajo_rendimiento_mes_siguiente")
    .eqNullSafe(F.col("objetivo_esperado"))
)

print("Etiquetas inconsistentes:", diferencias.count())
diferencias.show(20, truncate=False)

In [ ]:
import json
from datetime import datetime

reporte = {
    "fecha_verificacion": datetime.now().isoformat(),
    "tabla": "afp_analytics.tabla_analitica_mensual",
    "total_meses": final.count(),
    "fechas_unicas": final.select("fecha_mes").distinct().count(),
    "nulos": {
        "anio_ejercicio": final.filter(
            F.col("anio_ejercicio").isNull()
        ).count(),
        "afiliados_integra": final.filter(
            F.col("afiliados_integra").isNull()
        ).count()
    },
    "documentos_memorias": memorias.count(),
    "fechas_disponibilidad_nulas": nulos_disponibilidad,
    "memorias_futuras_detectadas": futuras.count(),
    "etiquetas_inconsistentes": diferencias.count(),
    "analisis_atipicos": resultados_atipicos
}

destino = BASE / "evidencias" / "11_verificaciones_finales.json"

with destino.open("w", encoding="utf-8") as archivo:
    json.dump(reporte, archivo, ensure_ascii=False, indent=4)

print("Reporte guardado en:", destino)